# Solving CartPole-v1 with PPO (from scratch)

This notebook demonstrates how to solve the CartPole-v1 environment using a custom implementation of the Proximal Policy Optimization (PPO) algorithm in PyTorch. No external RL libraries are used.

---

**Note:** All environment and hyperparameter settings are now collected in a single `CONFIG` dictionary at the top of the notebook. To change the environment or any hyperparameter, simply edit the values in the config cell.

## 1. Install and Import Required Libraries
We will use gymnasium and torch for this implementation.

In [1]:
# Install required packages
%pip install stable-baselines3 gymnasium tsilva-notebook-utils --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

In [3]:
from wandb import login
login()

wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2. Set Up CartPole-v1 Environment
We will initialize the CartPole-v1 environment and display its basic information.

In [4]:
ENV_ID = "CartPole-v1"
#ENV_ID = "Acrobot-v1"
#ENV_ID = "LunarLander-v3"
#ENV_ID = "Pendulum-v1"
#ENV_ID = "MountainCar-v0"

In [5]:
import gymnasium as gym
import torch
import torch.nn as nn
import numpy as np
import platform, os
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.utils import set_random_seed

# Set device for training

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# --- Config dictionary for all hyperparameters and environment settings ---
def setup_config(env_id):
    common = dict(
        env_id=env_id,         # Environment name
        seed=42,               # Random seed for reproducibility
        gamma=0.99,            # Discount factor for future rewards
        lam=0.95,              # GAE lambda for advantage estimation
        clip_epsilon=0.2,      # PPO clip range for policy update
        minibatch_size=64,     # Minibatch size for SGD
        episodes_per_epoch=20, # Number of episodes per training epoch
        eval_interval=5,       # Evaluate every N epochs
        eval_episodes=20,      # Number of episodes for evaluation
        reward_threshold=200,  # Reward threshold to consider environment solved
        policy_lr=3e-4,        # Learning rate for policy network
        value_lr=1e-3,         # Learning rate for value network
        hidden_dim=64,         # Hidden layer size for networks
        entropy_coef=0.01,     # Coefficient for entropy bonus (encourages exploration)
        normalize=False,       # Whether to use input normalization
        max_envs=None          # Maximum number of parallel environments
    )
    env_specific = {
        "CartPole-v1": dict(
            gamma=0.99,           # Standard discount for CartPole
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for CartPole
            minibatch_size=32,    # Smaller batch for faster updates
            episodes_per_epoch=16,# Fewer episodes per epoch for quick feedback
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=475, # Official CartPole-v1 solved threshold
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for CartPole
            entropy_coef=0.01     # Typical entropy for CartPole
        ),
        "LunarLander-v3": dict(
            gamma=0.99,           # Standard discount for LunarLander
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for LunarLander
            minibatch_size=64,    # Larger batch for more stable updates
            episodes_per_epoch=8, # Fewer episodes per epoch (env is longer)
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=200, # Solved threshold for LunarLander-v3
            policy_lr=1e-4,       # Lower LR for more complex env
            value_lr=5e-4,        # Lower LR for value net
            hidden_dim=128,       # Larger net for more complex env
            entropy_coef=0.02     # Higher entropy for more exploration
        ),
        "Acrobot-v1": dict(
            gamma=0.99,           # Standard discount for Acrobot
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Acrobot
            minibatch_size=32,    # Smaller batch for faster updates
            episodes_per_epoch=16,# Fewer episodes per epoch for quick feedback
            eval_interval=2,      # Evaluate more frequently for fast convergence
            eval_episodes=10,     # Fewer eval episodes for speed
            reward_threshold=-100, # Solved threshold for Acrobot-v1 (average reward > -100)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=64,        # Small net is sufficient for Acrobot
            entropy_coef=0.01     # Typical entropy for Acrobot
        ),
        "Pendulum-v1": dict(
            gamma=0.99,           # Standard discount for Pendulum
            lam=0.95,             # GAE lambda, balances bias/variance
            clip_epsilon=0.2,     # PPO clip range, stable for Pendulum
            minibatch_size=64,    # Larger batch for continuous action
            episodes_per_epoch=8, # Fewer episodes per epoch (env is longer)
            eval_interval=2,      # Evaluate every 2 epochs
            eval_episodes=5,      # Fewer eval episodes for speed
            reward_threshold=-200, # Solved threshold for Pendulum-v1 (average reward > -200)
            policy_lr=3e-4,       # Standard PPO learning rate
            value_lr=1e-3,        # Slightly higher for value net
            hidden_dim=128,       # Larger net for continuous control
            entropy_coef=0.0      # No entropy for deterministic continuous control
        ),
        "MountainCar-v0": dict(
            gamma=0.99,             # Discount factor (keep)
            lam=0.97,               # Slightly higher GAE lambda for more bias reduction
            clip_epsilon=0.15,      # Tighter PPO clip for more stable updates
            minibatch_size=16,      # Smaller minibatch for more frequent updates
            episodes_per_epoch=24,  # More episodes per epoch for better sampling
            eval_interval=2,        # Keep frequent evaluation
            eval_episodes=10,       # Keep
            reward_threshold=-110,  # Keep
            policy_lr=1e-4,         # Lower learning rate for more stable policy updates
            value_lr=5e-4,          # Lower value net LR for stability
            hidden_dim=128,         # Larger network for more capacity
            entropy_coef=0.05       # Higher entropy for hard exploration
        ),
    }
    if env_id not in env_specific:
        raise ValueError(f"Unsupported env_id: {env_id}")
    return {**common, **env_specific[env_id]}

# Set normalize=True to enable input normalization
CONFIG = setup_config(ENV_ID)

# --- Runtime metadata --------------------------------------------------------
import subprocess, torch, platform, os

def _get_git_commit() -> str:
    """Return the short SHA if this is a Git repo, else 'unknown'."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:          # not a Git checkout or Git not installed
        return "unknown"

def runtime_metadata():
    return {
        "git_commit": _get_git_commit(),
        "torch_version": torch.__version__,
        "torch_cuda": torch.version.cuda or "cpu",
        "cuda_device": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
        ),
        "python_version": platform.python_version(),
        "run_host": os.uname().nodename,
    }


# --- Unified build_env function for all env construction (SB3 style, with subproc option) ---
def _build_env(env_id, normalize=False, n_envs=1, seed=None):
    vec_env_cls = SubprocVecEnv if n_envs > 1 else DummyVecEnv
    env = make_vec_env(env_id, n_envs=n_envs, seed=seed, vec_env_cls=vec_env_cls)
    if normalize: env = VecNormalize(env, norm_obs=True, norm_reward=False)
    return env

# Set up the environment with SB3 wrappers if requested
CONFIG = setup_config(ENV_ID)
CONFIG.update(runtime_metadata())

set_random_seed(CONFIG['seed'], using_cuda=torch.cuda.is_available())

import multiprocessing
N_ENVS = min(multiprocessing.cpu_count(), CONFIG['max_envs']) if CONFIG['max_envs'] is not None else multiprocessing.cpu_count()
build_env = lambda seed: _build_env(CONFIG['env_id'], normalize=CONFIG['normalize'], n_envs=N_ENVS, seed=seed)
env = build_env(CONFIG['seed'])
obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.n
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

Using device: cuda
Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Action space: Discrete(2)


## 3. Implement PPO Agent
We will define the policy and value networks, and the PPO update step.

In [6]:
# Policy and Value Networks
class PolicyNet(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, act_dim)
        )
    def forward(self, x):
        return self.net(x)

class ValueNet(nn.Module):
    def __init__(self, obs_dim, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CONFIG['hidden_dim']
        self.net = nn.Sequential(
            nn.Linear(obs_dim, h), nn.Tanh(),
            nn.Linear(h, h), nn.Tanh(),
            nn.Linear(h, 1)
        )
    def forward(self, x):
        return self.net(x)

## 4. Train PPO Agent
We will train the PPO agent on CartPole-v1.

In [49]:
from __future__ import annotations
from typing import List, Tuple
import numpy as np
import torch
from torch.distributions import Categorical


# --------------------------------------------------------------------- #
# Helpers                                                               #
# --------------------------------------------------------------------- #
def _get_device(policy):
    """Return the device hosting *policy* parameters, else CPU."""
    try:
        return next(policy.parameters()).device
    except AttributeError:                         # plain Callable
        return torch.device("cpu")


# --------------------------------------------------------------------- #
# Roll-out routine                                                      #
# --------------------------------------------------------------------- #
def collect_rollouts(
    env,
    policy_model,
    value_model,
    n_episodes: int = 1,
    deterministic: bool = False,
    render: bool = False,
    collect_frames: bool = False
) -> Tuple[
    List[float],                                   # total episode returns
    Tuple[List, List, List, List, List, List],     # trajectory buffers
    List[List[np.ndarray]] | None                  # 🔸 frames by episode
]:
    """
    Light-weight rollout collector that works with Gymnasium *and*
    SB3 vector environments and **guarantees** the number of returned
    episodes equals ``n_episodes``.

    When *collect_frames* is True, the third return value is now a
    ``List[List[np.ndarray]]``: one inner list per completed episode,
    preserving frame boundaries.
    """
    if n_episodes < 1:
        raise ValueError("n_episodes must be ≥ 1")

    device   = _get_device(policy_model)
    obs      = env.reset()
    n_envs   = env.num_envs

    # Per-env episode buffers
    ep_obs, ep_actions, ep_rewards = [[] for _ in range(n_envs)], [[] for _ in range(n_envs)], [[] for _ in range(n_envs)]
    ep_dones, ep_logps, ep_values  = [[] for _ in range(n_envs)], [[] for _ in range(n_envs)], [[] for _ in range(n_envs)]
    ep_frames                       = [[] for _ in range(n_envs)] if collect_frames else None

    # Global rollout buffers
    obs_buf, act_buf, rew_buf, done_buf, logp_buf, val_buf = ([] for _ in range(6))
    frames_buf: List[List[np.ndarray]] = [] if collect_frames else None     # 🔸 structured
    total_ep_returns: List[float] = []
    episodes_collected            = 0

    with torch.no_grad():
        while episodes_collected < n_episodes:
            # ------------------------------------------------ policy forward
            obs_t  = torch.as_tensor(obs, dtype=torch.float32, device=device)
            logits = policy_model(obs_t)
            dist   = Categorical(logits=logits)

            action_tensor = (
                dist.probs.argmax(dim=-1) if deterministic else dist.sample()
            )                           # shape (n_envs,)
            logp_tensor   = dist.log_prob(action_tensor)

            action_np = np.atleast_1d(action_tensor.cpu().numpy())
            logp_np   = np.atleast_1d(logp_tensor.cpu().numpy())
            value_np  = np.atleast_1d(value_model(obs_t).squeeze(-1).cpu().numpy())

            # ------------------------------------------------ env step
            step_out = env.step(action_np)
            # Support both 4- and 5-tuple returns
            if len(step_out) == 4:
                next_obs, reward, done, _ = step_out
            else:
                next_obs, reward, done, trunc, _ = step_out   # Gymnasium
                done = np.logical_or(done, trunc)

            reward = np.atleast_1d(reward)        # 1-D

            # -------- optional explicit reset for Gymnasium VecEnv ----
            if done.any() and hasattr(env, "reset_done"):
                reset_obs, *_ = env.reset_done()
                next_obs[done] = reset_obs
            # ----------------------------------------------------------

            # ------------------------------------------------ frame capture
            if collect_frames:
                frame_batch = env.get_images()
                assert len(frame_batch) == n_envs, \
                    "get_images() must return a list of length n_envs"
                for i in range(n_envs):
                    ep_frames[i].append(frame_batch[i])

            # ------------------------------------------------ store transition
            for i in range(n_envs):
                # Stop collecting as soon as quota is reached
                if episodes_collected >= n_episodes:
                    break

                ep_obs[i].append(obs[i])
                ep_actions[i].append(int(action_np[i]))
                ep_rewards[i].append(float(reward[i]))
                ep_dones[i].append(bool(done[i]))
                ep_logps[i].append(float(logp_np[i]))
                ep_values[i].append(float(value_np[i]))

                if done[i]:
                    # flush episode i
                    total_ep_returns.append(float(np.sum(ep_rewards[i])))

                    obs_buf.extend(ep_obs[i]);     ep_obs[i].clear()
                    act_buf.extend(ep_actions[i]); ep_actions[i].clear()
                    rew_buf.extend(ep_rewards[i]); ep_rewards[i].clear()
                    done_buf.extend(ep_dones[i]);  ep_dones[i].clear()
                    logp_buf.extend(ep_logps[i]);  ep_logps[i].clear()
                    val_buf.extend(ep_values[i]);  ep_values[i].clear()

                    if collect_frames:
                        # 🔸 preserve episode boundary
                        frames_buf.append(ep_frames[i].copy())
                        ep_frames[i].clear()

                    episodes_collected += 1

            if render:
                env.render()

            obs = next_obs                        # iterate

    # ------------------------------ return buffers
    if collect_frames:
        return (
            total_ep_returns,
            (obs_buf, act_buf, rew_buf, done_buf, logp_buf, val_buf),
            frames_buf,
        )

    return total_ep_returns, (
        obs_buf,
        act_buf,
        rew_buf,
        done_buf,
        logp_buf,
        val_buf,
    )


# --------------------------------------------------------------------- #
# Simple evaluation helper                                              #
# --------------------------------------------------------------------- #
def evaluate_policy(policy, *, n_episodes: int, seed: int | None = None):
    """
    Convenience wrapper: build a fresh env, run deterministic episodes,
    return (mean_return, list_of_returns).

    A ``build_env(seed)`` factory must be defined by the caller and must
    *return a vector environment*.
    """
    seed = seed if seed is not None else np.random.randint(0, 1_000_000)
    env  = build_env(seed)
    ep_returns, _ = collect_rollouts(
        env,
        policy_model=policy,
        value_model=lambda x: torch.zeros(len(x)),  # dummy critic
        n_episodes=n_episodes,
        deterministic=True,
    )
    return float(np.mean(ep_returns)), ep_returns




In [8]:
policy_model = PolicyNet(obs_dim, act_dim)
value_model = ValueNet(obs_dim)

In [59]:
rewards, tuplex, frames = collect_rollouts(
    env, 
    policy_model,
    value_model,
    n_episodes=8, 
    deterministic=True, 
    render=False, 
    collect_frames=True
)
rewards, len(frames)

([8.0, 9.0, 9.0, 9.0, 9.0, 9.0, 10.0, 10.0], 8)

In [60]:
frames_to_video(frames, fps=1, grid=(2, 2)) # yes it does, check bbf output too

ValueError: 8 episodes won't fit into a 2×2 grid.

In [ ]:
import random
build_env = lambda seed: _build_env(CONFIG['env_id'], normalize=CONFIG['normalize'], n_envs=2, seed=seed)
env = build_env(random.randint(0, 1_000_000))
obs = env.reset()

In [ ]:
env.step([random.randint(0, 1), random.randint(0, 1)])

In [52]:
import os
import shutil
import subprocess
import tempfile
from pathlib import Path
from typing import Iterable, Sequence

import numpy as np
import imageio.v3 as iio   # pip install imageio[ffmpeg]  (or just imageio)

def frames_to_video(
    frames: Iterable[np.ndarray] | Sequence[np.ndarray],
    *,
    fps: int = 30,
    out_path: str | os.PathLike | None = None,
    codec: str = "libx264",
    crf: int = 23,
    preset: str = "medium",
) -> Path:
    """
    Encode *frames* (H,W,3 uint8 RGB) into an MP4 file using FFmpeg.

    Parameters
    ----------
    frames   : iterable / sequence of np.ndarray
               Each array must be uint8 RGB with the *same* shape.
               You may pass a generator to avoid holding frames in memory.
    fps      : int, video frame-rate (default 30).
    out_path : str | Path | None.  If None, an `.mp4` file is created
               inside a temp directory and the full path is returned.
    codec    : str, FFmpeg video codec (default "libx264").
    crf      : int, quality factor for x264/265 (lower → better quality).
    preset   : str, x264/265 speed/efficiency preset.

    Returns
    -------
    Path to the encoded video.
    """
    # --------------------------------------------------------------- #
    # 1. Prep temp workspace                                          #
    # --------------------------------------------------------------- #
    tmp_root = tempfile.TemporaryDirectory()
    frames_dir = Path(tmp_root.name) / "frames"
    frames_dir.mkdir()

    # --------------------------------------------------------------- #
    # 2. Stream frames to PNGs on disk                                #
    # --------------------------------------------------------------- #
    for idx, frame in enumerate(frames):
        if not (isinstance(frame, np.ndarray) and frame.ndim == 3 and frame.dtype == np.uint8):
            raise ValueError(f"Frame {idx} is not an RGB uint8 ndarray of shape (H, W, 3).")
        iio.imwrite(frames_dir / f"frame_{idx:06d}.png", frame, plugin="pillow")

    n_frames = idx + 1  # last idx from loop

    if n_frames == 0:
        raise ValueError("No frames provided.")

    # --------------------------------------------------------------- #
    # 3. Build and run FFmpeg command                                 #
    # --------------------------------------------------------------- #
    #out_path = Path(out_path) if out_path is not None else Path(tmp_root.name) / "episode.mp4"

    if out_path is None:                                  # durable temp file
        fd, tmp_name = tempfile.mkstemp(suffix=".mp4")
        os.close(fd)
        out_path = Path(tmp_name)


    cmd = [
        "ffmpeg",
        "-loglevel", "error",          # make it quiet unless there's a problem
        "-y",                          # overwrite existing file
        "-framerate", str(fps),
        "-i", str(frames_dir / "frame_%06d.png"),
        "-c:v", codec,
        "-preset", preset,
        "-crf", str(crf),
        "-pix_fmt", "yuv420p",         # wider player compatibility
        str(out_path)
    ]

    subprocess.run(cmd, check=True)

    # --------------------------------------------------------------- #
    # 4. Clean up frame PNGs and keep/return the video path           #
    # --------------------------------------------------------------- #
    shutil.rmtree(frames_dir)          # only the individual PNGs
    # tmp_root itself is kept alive for the lifetime of tmp_root obj;
    # if out_path was inside tmp_root, caller keeps the Path until move/copy.

    from IPython.display import Video

    return Video(out_path, embed=True, width=480) 
from __future__ import annotations
import os, shutil, subprocess, tempfile
from pathlib import Path
from collections.abc import Iterable, Sequence

import numpy as np
import imageio.v3 as iio
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Video

def frames_to_video(
    frames: Iterable[np.ndarray] | Iterable[Iterable[np.ndarray]],
    *, fps: int = 30, out_path=None,
    codec: str = "libx264", crf: int = 23, preset: str = "medium",
    font: ImageFont.ImageFont | None = None,
    text_xy: tuple[int, int] = (5, 5),
    text_color: tuple[int, int, int] = (255, 255, 255),
    stroke_color: tuple[int, int, int] = (0, 0, 0),
    stroke_width: int = 1,
) -> Video:
    """
    Encode a list/iterator of frames **OR** list-of-lists (episodes) to MP4.

    * Flat input  -> label 'Step: <idx>'
    * Nested      -> label 'Ep: <ep>  Step: <idx>'
    """
    if font is None:
        font = ImageFont.load_default()

    # --------------------------------------------------------------- #
    # Decide whether we have episodes (nested) or single sequence     #
    # (We peek at the first element with minimal memory impact.)      #
    # --------------------------------------------------------------- #
    try:
        first = next(iter(frames))          # may raise StopIteration
    except StopIteration:
        raise ValueError("No frames provided.")

    is_episode_mode = isinstance(first, (Sequence, Iterable)) and not (
        isinstance(first, np.ndarray) and first.ndim == 3
    )

    # We need an iterator that yields (episode_idx, step_idx, frame)
    def frame_stream():
        if is_episode_mode:
            for ep_idx, episode in enumerate(frames):
                for step_idx, frame in enumerate(episode):
                    yield ep_idx, step_idx, frame
        else:
            for step_idx, frame in enumerate(frames):
                yield 0, step_idx, frame  # ep_idx is always 0 for flat
    # --------------------------------------------------------------- #
    # Temp dir for PNGs                                               #
    # --------------------------------------------------------------- #
    tmp_root = tempfile.TemporaryDirectory()
    frames_dir = Path(tmp_root.name) / "frames"
    frames_dir.mkdir()

    n_frames = 0
    for ep_idx, step_idx, frame in frame_stream():
        if not (
            isinstance(frame, np.ndarray)
            and frame.ndim == 3
            and frame.dtype == np.uint8
        ):
            raise ValueError(
                f"Frame {step_idx} (episode {ep_idx}) is not uint8 RGB (H,W,3)."
            )

        # ---------------- label ---------------- #
        img = Image.fromarray(frame)
        draw = ImageDraw.Draw(img, "RGB")
        if is_episode_mode:
            label = f"Ep: {ep_idx}  Step: {step_idx}"
        else:
            label = f"Step: {step_idx}"
        draw.text(
            text_xy,
            label,
            font=font,
            fill=text_color,
            stroke_fill=stroke_color,
            stroke_width=stroke_width,
        )
        iio.imwrite(
            frames_dir / f"frame_{n_frames:06d}.png",
            np.asarray(img),
            plugin="pillow",
        )
        n_frames += 1

    if n_frames == 0:
        raise ValueError("No frames provided.")

    # --------------------------------------------------------------- #
    # Permanent output path                                           #
    # --------------------------------------------------------------- #
    if out_path is None:
        fd, tmp_name = tempfile.mkstemp(suffix=".mp4")
        os.close(fd)
        out_path = Path(tmp_name)
    else:
        out_path = Path(out_path)

    # --------------------------------------------------------------- #
    # Encode with FFmpeg                                              #
    # --------------------------------------------------------------- #
    cmd = [
        "ffmpeg",
        "-loglevel",
        "error",
        "-y",
        "-framerate",
        str(fps),
        "-i",
        str(frames_dir / "frame_%06d.png"),
        "-c:v",
        codec,
        "-preset",
        preset,
        "-crf",
        str(crf),
        "-pix_fmt",
        "yuv420p",
        str(out_path),
    ]
    subprocess.run(cmd, check=True)

    shutil.rmtree(frames_dir)  # clean up PNGs; tmp_root dies automatically
    return Video(str(out_path), embed=True, width=480)

from __future__ import annotations
import os, shutil, subprocess, tempfile
from pathlib import Path
from collections.abc import Iterable, Sequence
import itertools

import numpy as np
import imageio.v3 as iio
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Video

# ---------------------------------------------------------------------
#  Main entry point
# ---------------------------------------------------------------------
def frames_to_video(
    frames: Iterable[np.ndarray] | Iterable[Iterable[np.ndarray]],
    *,
    fps: int = 30,
    out_path: str | os.PathLike | None = None,
    codec: str = "libx264",
    crf: int = 23,
    preset: str = "medium",
    #
    # --- overlay text styling ---------------------------------------
    font: ImageFont.ImageFont | None = None,
    text_xy: tuple[int, int] = (5, 5),
    text_color: tuple[int, int, int] = (255, 255, 255),
    stroke_color: tuple[int, int, int] = (0, 0, 0),
    stroke_width: int = 1,
    #
    # --- grid mode ---------------------------------------------------
    grid: tuple[int, int] | None = None,   # e.g. (2, 3) for 2 rows × 3 cols
) -> Video:
    """
    Encode frames to MP4.

    Parameters
    ----------
    frames : iterable of np.ndarray (flat)  *or*
             iterable of iterable (episodes)
    grid   : (rows, cols) to lay out multiple episodes simultaneously.
             Must be omitted for a flat list of frames.  When provided,
             `frames` *must* be a nested iterable whose length ≤ rows*cols.

    Returns
    -------
    IPython.display.Video object pointing at the encoded file.
    """
    # -----------------------------------------------------------------
    # Font default
    # -----------------------------------------------------------------
    if font is None:
        font = ImageFont.load_default()

    # -----------------------------------------------------------------
    # Detect flat vs episodes
    # -----------------------------------------------------------------
    frames_iter = iter(frames)
    try:
        first = next(frames_iter)
    except StopIteration:
        raise ValueError("No frames provided.")
    frames_iter = itertools.chain([first], frames_iter)  # put it back

    episode_mode = isinstance(first, (Sequence, Iterable)) and not (
        isinstance(first, np.ndarray) and first.ndim == 3
    )

    if grid is not None and not episode_mode:
        raise ValueError("`grid=` only makes sense when `frames` is a list of episodes.")
    if episode_mode and grid is None:
        # Allow old behaviour (episodes one after another, not grid)
        pass

    # -----------------------------------------------------------------
    # Standardise data: produce a list of episodes, each a list of frames
    # (If input was flat, wrap it as one episode.)
    # -----------------------------------------------------------------
    if episode_mode:
        episodes = list(frames_iter)
    else:
        episodes = [list(frames_iter)]

    # Ensure every frame is at least validated once & record shape
    def validate_frame(frame, msg=""):
        if not (isinstance(frame, np.ndarray) and frame.ndim == 3 and frame.dtype == np.uint8):
            raise ValueError(f"Frame {msg} is not uint8 RGB with shape (H, W, 3).")

    first_valid_frame = None
    for ep_idx, ep in enumerate(episodes):
        if not ep:
            raise ValueError(f"Episode {ep_idx} is empty.")
        for st_idx, fr in enumerate(ep):
            validate_frame(fr, f"{st_idx} in episode {ep_idx}")
        if first_valid_frame is None:
            first_valid_frame = episodes[ep_idx][0]

    H, W, _ = first_valid_frame.shape

    # -----------------------------------------------------------------
    # Grid bookkeeping
    # -----------------------------------------------------------------
    if grid is None:
        grid_rows, grid_cols = 1, 1
    else:
        grid_rows, grid_cols = grid
        max_slots = grid_rows * grid_cols
        if len(episodes) > max_slots:
            raise ValueError(f"{len(episodes)} episodes won't fit into a {grid_rows}×{grid_cols} grid.")
    # Map episode index ➜ (row, col)
    ep_to_cell = {
        ep_idx: divmod(ep_idx, grid_cols)   # (row, col) in row-major order
        for ep_idx in range(len(episodes))
    }

    # Total number of composite frames = longest episode length
    max_steps = max(len(ep) for ep in episodes)

    # -----------------------------------------------------------------
    # Temp dir to store PNGs
    # -----------------------------------------------------------------
    tmp_root = tempfile.TemporaryDirectory()
    frames_dir = Path(tmp_root.name) / "frames"
    frames_dir.mkdir()

    # -----------------------------------------------------------------
    # Helper: stamp label onto a PIL.Image in-place
    # -----------------------------------------------------------------
    def stamp(img: Image.Image, label: str):
        draw = ImageDraw.Draw(img, "RGB")
        draw.text(
            text_xy,
            label,
            font=font,
            fill=text_color,
            stroke_fill=stroke_color,
            stroke_width=stroke_width,
        )

    # -----------------------------------------------------------------
    # Write composite frames
    # -----------------------------------------------------------------
    frame_counter = 0
    for step_idx in range(max_steps):
        if grid_rows == grid_cols == 1:
            # ---------- Simple (not grid) case -----------------------
            frame = episodes[0][step_idx] if step_idx < len(episodes[0]) else episodes[0][-1]
            img = Image.fromarray(frame)
            stamp(img, f"Step: {step_idx}" if not episode_mode else f"Ep: 0  Step: {step_idx}")
            iio.imwrite(
                frames_dir / f"frame_{frame_counter:06d}.png",
                np.asarray(img),
                plugin="pillow",
            )
            frame_counter += 1
            continue

        # ---------- Grid case ---------------------------------------
        composite = np.zeros((grid_rows * H, grid_cols * W, 3), dtype=np.uint8)

        for ep_idx, episode in enumerate(episodes):
            row, col = ep_to_cell[ep_idx]
            y0, y1 = row * H, (row + 1) * H
            x0, x1 = col * W, (col + 1) * W

            if step_idx < len(episode):
                src = episode[step_idx]
            else:
                # Episode finished → keep last frame frozen
                src = episode[-1]

            img = Image.fromarray(src)
            stamp(img, f"Ep: {ep_idx}  Step: {step_idx}")
            composite[y0:y1, x0:x1] = np.asarray(img)

        iio.imwrite(
            frames_dir / f"frame_{frame_counter:06d}.png",
            composite,
            plugin="pillow",
        )
        frame_counter += 1

    # -----------------------------------------------------------------
    # Permanent output path
    # -----------------------------------------------------------------
    if out_path is None:
        fd, tmp_name = tempfile.mkstemp(suffix=".mp4")
        os.close(fd)
        out_path = Path(tmp_name)
    else:
        out_path = Path(out_path)

    # -----------------------------------------------------------------
    # Encode with FFmpeg
    # -----------------------------------------------------------------
    cmd = [
        "ffmpeg", "-loglevel", "error", "-y",
        "-framerate", str(fps),
        "-i", str(frames_dir / "frame_%06d.png"),
        "-c:v", codec, "-preset", preset, "-crf", str(crf),
        "-pix_fmt", "yuv420p",
        str(out_path),
    ]
    subprocess.run(cmd, check=True)

    shutil.rmtree(frames_dir)
    return Video(str(out_path), embed=True, width=640)   # width here is just the notebook embed size



In [40]:
build_env = lambda seed: _build_env(CONFIG['env_id'], normalize=CONFIG['normalize'], n_envs=2, seed=seed)
env  = build_env(42)
policy_model = PolicyNet(obs_dim, act_dim)
value_model = ValueNet(obs_dim)
build_env = lambda seed: _build_env(CONFIG['env_id'], normalize=CONFIG['normalize'], n_envs=2, seed=seed)
ep_returns, trajectories, frames = collect_rollouts(env, policy_model, value_model, n_episodes=10, deterministic=True, collect_frames=True)
#frames_to_video(frames, fps=30, out_path="cartpole_run2.mp4")
trajectories[1]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [ ]:
env = make_vec_env("CartPole-v1", n_envs=2, seed=42, vec_env_cls=SubprocVecEnv)
env.reset()
env.step([env.action_space.sample() for _ in range(env.num_envs)])
# TODO: does env auto-reset
# TODO: done is not being calculated correctly (truncated is dict)

In [ ]:
env = make_vec_env("CartPole-v1", n_envs=2, seed=42, vec_env_cls=SubprocVecEnv)
env.reset()
frames = env.render()
len(frames), frames[0].shape, frames[1].shape, frames[2].shape, frames[3].shape, frames[4].shape

In [ ]:

len()

In [ ]:
# Helper functions for PPO
import random
import torch.optim as optim
import time  # <-- Ensure time is imported for wall time

def set_global_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    advantages = []
    gae = 0
    values = values + [0]
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * values[t+1] * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1 - dones[t]) * gae
        advantages.insert(0, gae)
    return advantages

# Vectorized GAE for parallel envs

def compute_gae_vec(rewards, values, dones, gamma=0.99, lam=0.95, n_envs=1):
    advantages = []
    start = 0
    for _ in range(n_envs):
        # Find episode length for this env
        ep_len = 0
        for j in range(start, len(rewards)):
            ep_len += 1
            if dones[j]:
                break
        ep_adv = compute_gae(rewards[start:start+ep_len], values[start:start+ep_len], dones[start:start+ep_len], gamma, lam)
        advantages.extend(ep_adv)
        start += ep_len
    return advantages

# Set fixed seed for training
set_global_seed(CONFIG['seed'])
env = build_env(CONFIG['seed'])
obs = env.reset()

# PPO Training Loop
entropy_coef = CONFIG.get('entropy_coef', 0.0)
policy_net = PolicyNet(obs_dim, act_dim, hidden_dim=CONFIG['hidden_dim']).to(device)
value_model = ValueNet(obs_dim, hidden_dim=CONFIG['hidden_dim']).to(device)
policy_optim = optim.Adam(policy_net.parameters(), lr=CONFIG['policy_lr'])
value_optim = optim.Adam(value_model.parameters(), lr=CONFIG['value_lr'])

clip_epsilon = CONFIG['clip_epsilon']
minibatch_size = CONFIG['minibatch_size']
gamma = CONFIG['gamma']
lam = CONFIG['lam']
episodes_per_epoch = CONFIG['episodes_per_epoch']
eval_interval = CONFIG['eval_interval']
eval_episodes = CONFIG['eval_episodes']
reward_threshold = CONFIG['reward_threshold']

train_rewards = []  # Store mean reward per epoch
solved = False
current_epoch = 0

start_time = time.time()  # <-- Start wall time measurement

while not solved:
    # Use collect_rollouts for training
    episode_rewards, (obs_buf, act_buf, rew_buf, done_buf, logp_buf, val_buf) = collect_rollouts(
        policy_net, env, n_episodes=episodes_per_epoch, value_model=value_model, deterministic=False, render=False)
    train_rewards.append(np.mean(episode_rewards))
    # Compute advantages per environment episode
    adv_buf = compute_gae_vec(rew_buf, val_buf, done_buf, gamma, lam, n_envs=N_ENVS)
    # Truncate val_buf to match adv_buf length (in case of partial episodes)
    val_buf_trunc = val_buf[:len(adv_buf)]
    ret_buf = (np.array(adv_buf) + np.array(val_buf_trunc)).tolist()
    # Convert buffers to tensors and move to device
    obs_t = torch.as_tensor(np.array(obs_buf[:len(adv_buf)]), dtype=torch.float32, device=device)
    act_tensor = torch.as_tensor(np.array(act_buf[:len(adv_buf)]), dtype=torch.int64, device=device)
    adv_tensor = torch.as_tensor(np.array(adv_buf), dtype=torch.float32, device=device)
    ret_tensor = torch.as_tensor(np.array(ret_buf), dtype=torch.float32, device=device)
    logp_old_tensor = torch.as_tensor(np.array(logp_buf[:len(adv_buf)]), dtype=torch.float32, device=device)
    # Normalize advantages
    adv_tensor = (adv_tensor - adv_tensor.mean()) / (adv_tensor.std() + 1e-8)
    # PPO update
    num_samples = len(obs_t)
    for _ in range(10):
        idx = np.random.permutation(num_samples)
        for start in range(0, num_samples, minibatch_size):
            end = min(start + minibatch_size, num_samples)
            mb_idx = idx[start:end]
            policy_logits = policy_net(obs_t[mb_idx])
            logits_dist = Categorical(logits=policy_logits)
            logp = logits_dist.log_prob(act_tensor[mb_idx])
            ratio = torch.exp(logp - logp_old_tensor[mb_idx])
            surr1 = ratio * adv_tensor[mb_idx]
            surr2 = torch.clamp(ratio, 1 - clip_epsilon, 1 + clip_epsilon) * adv_tensor[mb_idx]
            entropy = logits_dist.entropy().mean()
            policy_loss = -torch.min(surr1, surr2).mean() - entropy_coef * entropy
            value_pred = value_model(obs_t[mb_idx]).squeeze()
            value_loss = ((ret_tensor[mb_idx] - value_pred) ** 2).mean()
            policy_optim.zero_grad()
            policy_loss.backward()
            policy_optim.step()
            value_optim.zero_grad()
            value_loss.backward()
            value_optim.step()
    current_epoch += 1
    print(f"Epoch {current_epoch} complete. Mean train reward: {train_rewards[-1]:.2f}")
    if current_epoch % eval_interval == 0:
        mean_eval_reward, _ = evaluate_policy(policy_net, n_episodes=eval_episodes)
        print(f"Evaluation after {current_epoch} epochs: Mean reward = {mean_eval_reward:.2f}")
        if mean_eval_reward >= reward_threshold:
            print(f"Solved! Mean evaluation reward {mean_eval_reward:.2f} >= {reward_threshold}")
            solved = True

end_time = time.time()  # <-- End wall time measurement
print(f"Total training wall time: {end_time - start_time:.2f} seconds")

In [ ]:
# Plot training rewards after training
import matplotlib.pyplot as plt
plt.plot(train_rewards, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Mean Episode Reward')
plt.title('Training Reward per Epoch')
plt.grid(True)
plt.show()

In [ ]:
# Evaluate the trained policy_net on 100 episodes
mean_reward, episode_rewards = evaluate_policy(policy_net, n_episodes=100)
print(f"Mean reward over 100 episodes: {mean_reward:.2f}")

In [ ]:
from tsilva_notebook_utils.gymnasium import render_episode

# Use the trained policy_net and provide a callable that accepts env_kwargs and sets render_mode via gym.make

def make_env_for_render(env_kwargs=None):
    env_id = CONFIG['env_id']
    normalize = CONFIG.get('normalize', False)
    seed = CONFIG['seed']
    n_envs = 1
    use_subproc = False
    # Use gymnasium directly to set render_mode
    import gymnasium as gym
    render_mode = env_kwargs['render_mode'] if env_kwargs and 'render_mode' in env_kwargs else None
    env = gym.make(env_id, render_mode=render_mode)
    env.reset(seed=seed)
    if normalize:
        # VecNormalize expects a vectorized env, so skip normalization for rendering
        pass
    obs = env.reset()
    info = {}
    return env, obs, info

render_episode(
    env=make_env_for_render,
    model=policy_net
)

- TODO: Consider using RolloutBuffer
- TODO: Adapt for Torch Lightning